# Notebook 10 — RAG System Evaluation

This notebook evaluates ONLY end-to-end RAG answer quality.

It does NOT calculate Dense, BM25, Hybrid/RRF, or Cross-Encoder retrieval metrics.

Flow:
Frozen 26 questions → existing hybrid retrieval + reranking → Gemini RAG answer
→ Gemini reference answer from gold chunks → Gemini judge → final RAG metrics.

The existing benchmark, chunk metadata, vector store, and output paths are reused.
No new project structure is created.

Reference answers are generated only from the gold source chunks and are stored
with checkpointing so the notebook can resume after a rate-limit interruption.


In [1]:
# ============================================================
# SETUP AND EXISTING PROJECT PATHS
# ============================================================

import json
import os
import re
import pickle
import time
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import faiss

from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
OUTPUT_DIR = DATA_DIR / "outputs"

EVALUATION_SET_PATH = OUTPUT_DIR / "evaluation_set.json"
RAG_RESULTS_PATH = OUTPUT_DIR / "rag_evaluation_results.json"
REFERENCE_ANSWERS_PATH = OUTPUT_DIR / "reference_answers.json"
RAG_METRICS_PATH = OUTPUT_DIR / "rag_evaluation_metrics.json"
RAG_MATRIX_PATH = OUTPUT_DIR / "rag_evaluation_matrix.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)


c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\MD RAJA\Quant_Research_Intelligence_System
Output directory: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs


In [2]:
# ============================================================
# LOAD THE FROZEN 26-QUESTION BENCHMARK
# ============================================================

assert EVALUATION_SET_PATH.exists(), f"Missing: {EVALUATION_SET_PATH}"

with open(EVALUATION_SET_PATH, "r", encoding="utf-8") as f:
    evaluation_set = json.load(f)

assert len(evaluation_set) == 26, (
    f"Expected 26 questions, found {len(evaluation_set)}"
)

print("Frozen questions:", len(evaluation_set))
print("Categories:")

for k, v in Counter(x["category"] for x in evaluation_set).items():
    print(f"  {k}: {v}")


Frozen questions: 26
Categories:
  semantic_factual: 10
  formula_quantitative: 6
  table_based: 6
  figure_based: 2
  multi_chunk_reasoning: 1
  cross_document: 1


In [3]:
# ============================================================
# LOAD EXISTING PRODUCTION RETRIEVAL COMPONENTS
# Retrieval is only the input stage of end-to-end RAG evaluation.
# No retrieval metrics are calculated here.
# ============================================================

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

faiss_index = faiss.read_index(
    str(VECTOR_STORE_DIR / "quant_research.faiss")
)

with open(
    VECTOR_STORE_DIR / "quant_research_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)

chunk_lookup = {
    x["chunk_id"]: x
    for x in metadata
}

with open(
    VECTOR_STORE_DIR / "quant_research_bm25.pkl",
    "rb"
) as f:
    bm25 = pickle.load(f)

with open(
    VECTOR_STORE_DIR / "quant_research_bm25_ids.json",
    "r",
    encoding="utf-8"
) as f:
    bm25_ids = json.load(f)

print("FAISS records:", faiss_index.ntotal)
print("Metadata records:", len(metadata))
print("BM25 records:", len(bm25_ids))
print("Chunk lookup:", len(chunk_lookup))


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3363.77it/s]


FAISS records: 1141
Metadata records: 1141
BM25 records: 1132
Chunk lookup: 1141


In [4]:
def rrf_fusion(dense_ids, lexical_ids, k=60):
    scores = {}
    for rank, cid in enumerate(dense_ids, 1):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank)
    for rank, cid in enumerate(lexical_ids, 1):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def retrieve_and_rerank(query, dense_k=10, bm25_k=10, final_k=8):
    qv = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    _, dense_indices = faiss_index.search(qv, dense_k)
    dense_ids = [metadata[i]["chunk_id"] for i in dense_indices[0] if i >= 0]

    tokens = re.findall(r"\b\w+\b", query.lower())
    scores = bm25.get_scores(tokens)
    order = np.argsort(scores)[::-1][:bm25_k]
    lexical_ids = [bm25_ids[i] for i in order]

    fused = rrf_fusion(dense_ids, lexical_ids)
    hybrid_ids = [cid for cid, _ in fused[:10] if cid in chunk_lookup]

    pairs = [
        (query, chunk_lookup[cid].get("retrieval_text", chunk_lookup[cid].get("text", "")))
        for cid in hybrid_ids
    ]

    rerank_scores = reranker.predict(pairs)
    ranked = sorted(
        zip(hybrid_ids, rerank_scores),
        key=lambda x: float(x[1]),
        reverse=True
    )

    return [cid for cid, _ in ranked[:final_k]]


In [7]:
# ============================================================
# GEMINI CONFIGURATION
# ============================================================

import os
import time

from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types

env_path = find_dotenv()

if not env_path:
    raise FileNotFoundError("Could not find .env")

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLM_MODEL = os.getenv(
    "GEMINI_MODEL",
    "gemini-3.5-flash-lite"
)

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing from .env"
    )

client = genai.Client(
    api_key=GEMINI_API_KEY
)

# Rate-limit handling
RETRY_DELAYS = [10, 20, 40, 80]

# Delay between successful requests
SUCCESS_DELAY = 5

print("Gemini model:", LLM_MODEL)
print("Retry delays:", RETRY_DELAYS)
print("Success delay:", SUCCESS_DELAY)

Gemini model: gemini-2.5-flash-lite
Retry delays: [10, 20, 40, 80]
Success delay: 5


In [9]:
# ============================================================
# FIX GEMINI MODEL
# ============================================================

from dotenv import load_dotenv, find_dotenv
import os

env_path = find_dotenv()

if not env_path:
    raise FileNotFoundError(".env file not found")

load_dotenv(env_path, override=True)

LLM_MODEL = "gemini-3.5-flash-lite"

print("Gemini model:", LLM_MODEL)

Gemini model: gemini-3.5-flash-lite


## Reference answers

The frozen benchmark contains 26 questions and their gold source chunks.

This section generates a reference answer from ONLY those gold chunks using Gemini.
The generated reference answers are not human-verified answers; they are a
gold-context-based reference used for automated RAG evaluation.

Checkpointing is enabled. If Gemini is rate-limited, already generated answers
remain saved in `data/outputs/reference_answers.json` and the notebook can resume.


In [10]:
# ============================================================
# CREATE / RESUME REFERENCE ANSWERS FROM GOLD CHUNKS
# Gemini API + BATCHED EVALUATION: 10 → 10 → 6
# ============================================================

def build_gold_context(chunk_ids):

    records = []

    for chunk_id in chunk_ids:

        if chunk_id not in chunk_lookup:
            raise KeyError(
                f"Gold chunk not found in chunk_lookup: {chunk_id}"
            )

        chunk = chunk_lookup[chunk_id]

        text = (
            chunk.get("retrieval_text")
            or chunk.get("text")
            or ""
        )

        records.append(
            f"""
SOURCE CHUNK: {chunk_id}
SOURCE FILE: {chunk.get("source_file")}
PAGES: {chunk.get("pages", [])}
SECTION: {chunk.get("section_path", [])}

CONTENT:
{text}
"""
        )

    return "\n\n".join(records)


REFERENCE_PROMPT = """
You are creating a reference answer for evaluating a quantitative-finance RAG system.

Answer the QUESTION using ONLY the GOLD SOURCE CONTEXT.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. Include all important facts, numbers, formulas, comparisons,
   and relationships required by the question.
4. Preserve equations and numerical values accurately.
5. For table questions, report the relevant table values.
6. For figure questions, describe only what the supplied evidence supports.
7. For multi-chunk questions, combine information from the supplied chunks.
8. Give a concise but complete reference answer.
9. Do NOT include [S1], [S2] citations.
"""


# ------------------------------------------------------------
# Load existing checkpoint
# ------------------------------------------------------------

if REFERENCE_ANSWERS_PATH.exists():

    with open(
        REFERENCE_ANSWERS_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        reference_answers = json.load(f)

else:
    reference_answers = []


reference_by_id = {
    x["query_id"]: x
    for x in reference_answers
}

print("Existing reference answers:", len(reference_answers))


# ------------------------------------------------------------
# BATCH CONFIGURATION
# 26 questions → 10 + 10 + 6
# ------------------------------------------------------------

BATCH_SIZES = [10, 10, 6]

batch_ranges = [
    (1, 10),
    (11, 20),
    (21, 26),
]


# ------------------------------------------------------------
# Process batches
# ------------------------------------------------------------

for batch_number, (start, end) in enumerate(batch_ranges, 1):

    print("\n" + "=" * 80)
    print(f"BATCH {batch_number}: QUESTIONS {start} → {end}")
    print("=" * 80)

    batch_items = evaluation_set[start - 1:end]

    for local_index, item in enumerate(batch_items, start):

        query_id = item["query_id"]

        # ----------------------------------------------------
        # Resume from checkpoint
        # ----------------------------------------------------

        if (
            query_id in reference_by_id
            and reference_by_id[query_id].get(
                "reference_answer", ""
            ).strip()
        ):
            print(
                f"[{local_index}/26] "
                f"{query_id} — already saved, skipping."
            )
            continue

        # ----------------------------------------------------
        # Build GOLD context
        # ----------------------------------------------------

        gold_chunk_ids = [
            x["chunk_id"]
            for x in item["relevant_chunks"]
        ]

        gold_context = build_gold_context(gold_chunk_ids)

        print("\n" + "-" * 70)
        print(f"[{local_index}/26] {query_id}")
        print("Generating reference answer...")

        try:

            prompt = f"""
QUESTION:
{item["query"]}

GOLD SOURCE CONTEXT:
{gold_context}

Write the reference answer now.
"""

            answer = generate_with_gemini(
                prompt=prompt,
                system_instruction=REFERENCE_PROMPT,
                temperature=0.0,
                max_output_tokens=900,
            )

            # ------------------------------------------------
            # Save successful answer
            # ------------------------------------------------

            record = {
                "query_id": query_id,
                "category": item["category"],
                "query": item["query"],
                "reference_answer": answer,
                "gold_source_chunks": gold_chunk_ids,
            }

            reference_by_id[query_id] = record

            reference_answers = [
                reference_by_id[x["query_id"]]
                for x in evaluation_set
                if x["query_id"] in reference_by_id
            ]

            with open(
                REFERENCE_ANSWERS_PATH,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    reference_answers,
                    f,
                    indent=2,
                    ensure_ascii=False
                )

            print("Reference answer generated and saved.")

            # Small delay between successful requests
            if local_index < end:
                time.sleep(SUCCESS_DELAY)

        except RuntimeError as e:

            print("\nSTOPPED:", e)
            print(
                "All successfully generated answers have been saved."
            )
            print(
                "Run this same cell later to resume from the checkpoint."
            )

            raise


    # --------------------------------------------------------
    # Batch checkpoint
    # --------------------------------------------------------

    with open(
        REFERENCE_ANSWERS_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            [
                reference_by_id[x["query_id"]]
                for x in evaluation_set
                if x["query_id"] in reference_by_id
            ],
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n" + "=" * 80)
    print(f"BATCH {batch_number} COMPLETE: QUESTIONS {start} → {end}")
    print("=" * 80)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

with open(
    REFERENCE_ANSWERS_PATH,
    "r",
    encoding="utf-8"
) as f:
    reference_answers = json.load(f)

reference_by_id = {
    x["query_id"]: x
    for x in reference_answers
}

missing = [
    x["query_id"]
    for x in evaluation_set
    if not reference_by_id.get(
        x["query_id"],
        {}
    ).get(
        "reference_answer",
        ""
    ).strip()
]

print("\n" + "=" * 80)
print("REFERENCE ANSWER GENERATION SUMMARY")
print("=" * 80)

print("Reference answers saved:", len(reference_answers))
print("Missing:", len(missing))

if missing:
    print("Missing query IDs:", missing)
    print(
        "\nReference generation is incomplete. "
        "Run this cell again later to resume."
    )
else:
    print("\nAll 26 reference answers are ready.")

Existing reference answers: 0

BATCH 1: QUESTIONS 1 → 10

----------------------------------------------------------------------
[1/26] SEM01
Generating reference answer...
Reference answer generated and saved.

----------------------------------------------------------------------
[2/26] SEM02
Generating reference answer...
Reference answer generated and saved.

----------------------------------------------------------------------
[3/26] SEM03
Generating reference answer...
Reference answer generated and saved.

----------------------------------------------------------------------
[4/26] SEM04
Generating reference answer...
Reference answer generated and saved.

----------------------------------------------------------------------
[5/26] SEM05
Generating reference answer...
Reference answer generated and saved.

----------------------------------------------------------------------
[6/26] SEM06
Generating reference answer...
Reference answer generated and saved.

------------------

In [11]:
# ============================================================
# RUN END-TO-END RAG FOR ALL 26 QUESTIONS
# No retrieval metrics are calculated.
# Checkpoint after every successful question.
# ============================================================

if not REFERENCE_ANSWERS_PATH.exists():
    raise FileNotFoundError(
        "Reference answers are not available yet. "
        "Run the previous cell first."
    )

with open(
    REFERENCE_ANSWERS_PATH,
    "r",
    encoding="utf-8"
) as f:
    reference_answers = json.load(f)

if len(reference_answers) < 26:
    raise ValueError(
        f"Reference answers incomplete: {len(reference_answers)}/26"
    )


if RAG_RESULTS_PATH.exists():

    with open(
        RAG_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        rag_results = json.load(f)

else:
    rag_results = []


rag_by_id = {
    x["query_id"]: x
    for x in rag_results
}

print("Existing RAG results:", len(rag_results))


for i, item in enumerate(evaluation_set, 1):

    qid = item["query_id"]

    if (
        qid in rag_by_id
        and rag_by_id[qid].get("status") == "success"
    ):
        print(f"[{i}/26] {qid} — already saved, skipping.")
        continue

    print("=" * 75)
    print(f"[{i}/26] {qid} — {item['category']}")

    try:

        retrieved_ids = retrieve_and_rerank(
            item["query"],
            10,
            10,
            8
        )

        context_records = build_context_records(
            retrieved_ids
        )

        context = build_context(
            context_records
        )

        response = generate_grounded_answer(
            item["query"],
            context
        )

        trace = build_traceability(
            response.get("answer", ""),
            context_records
        )

        result = {
            "query_id": qid,
            "category": item["category"],
            "query": item["query"],
            "retrieved_chunks": retrieved_ids,
            "answer": response.get("answer", ""),
            "status": response.get("status", "unknown"),
            "citations": response.get("citations", []),
            "traceability": trace,
            "guardrail": response.get("guardrail", {}),
            "llm_model": LLM_MODEL,
            "timestamp": datetime.now().isoformat(),
        }

        rag_by_id[qid] = result

        rag_results = [
            rag_by_id[x["query_id"]]
            for x in evaluation_set
            if x["query_id"] in rag_by_id
        ]

        with open(
            RAG_RESULTS_PATH,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                rag_results,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            "Status:",
            result["status"],
            "| Citations:",
            result["citations"]
        )

        if i < len(evaluation_set):
            time.sleep(SUCCESS_DELAY)

    except RuntimeError as e:

        print("\nSTOPPED:", e)
        print("Already generated RAG results have been saved.")
        print("Run this same cell later to resume.")
        break


print("\nRAG results saved:", len(rag_results))


Existing RAG results: 26
[1/26] SEM01 — semantic_factual
Status: success | Citations: ['S1', 'S2', 'S6']
[2/26] SEM02 — semantic_factual
Status: success | Citations: []
[3/26] SEM03 — semantic_factual
Status: success | Citations: ['S4']
[4/26] SEM04 — semantic_factual
Status: success | Citations: ['S2']
[5/26] SEM05 — semantic_factual
Status: success | Citations: []
[6/26] SEM06 — semantic_factual
Status: success | Citations: ['S1']
[7/26] SEM07 — semantic_factual
Status: success | Citations: ['S1', 'S2', 'S3']
[8/26] SEM08 — semantic_factual
Status: success | Citations: ['S2']
[9/26] SEM09 — semantic_factual
Status: success | Citations: ['S2']
[10/26] SEM10 — semantic_factual
Status: success | Citations: ['S2', 'S3']
[11/26] FOR01 — formula_quantitative
Status: success | Citations: []
[12/26] FOR02 — formula_quantitative
Status: success | Citations: ['S2']
[13/26] FOR03 — formula_quantitative
Status: success | Citations: ['S1', 'S2', 'S6']
[14/26] FOR04 — formula_quantitative
Status: 

In [13]:
# ============================================================
# LLM-AS-JUDGE
# ============================================================

JUDGE_SYSTEM_PROMPT = """
You are a strict evaluator of a quantitative-finance RAG system.

Evaluate the GENERATED ANSWER against:
1. REFERENCE ANSWER = answer generated only from gold source chunks.
2. RETRIEVED CONTEXT = evidence available to the RAG system.

Score each dimension from 0 to 5.

CORRECTNESS:
5 fully correct; 4 essentially correct; 3 partially correct;
2 substantial errors/omissions; 1 mostly incorrect; 0 incorrect/irrelevant.

COMPLETENESS:
5 covers all important gold points; 4 nearly all; 3 main idea;
2 partial; 1 minimal; 0 none.

FAITHFULNESS:
5 all substantive claims supported by context; 4 almost all;
3 mixed; 2 several unsupported; 1 mostly unsupported; 0 fabricated/contradictory.

CITATION_ACCURACY:
5 citations correctly support claims; 4 almost all; 3 mixed;
2 several weak/incorrect; 1 mostly unsupported; 0 unusable.

Do not reward writing style.
Do not use outside knowledge.
Return JSON only.
"""


def parse_json_response(text):

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    ).strip()

    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if not match:
        raise ValueError("Judge did not return JSON.")

    return json.loads(match.group(0))


def judge_rag_answer(
    query,
    reference_answer,
    generated_answer,
    context,
    citations
):

    prompt = f"""
QUESTION:
{query}

REFERENCE ANSWER:
{reference_answer}

GENERATED ANSWER:
{generated_answer}

GENERATED CITATIONS:
{citations}

RETRIEVED CONTEXT:
{context}

Return exactly:

{{
  "correctness": 0,
  "completeness": 0,
  "faithfulness": 0,
  "citation_accuracy": 0,
  "overall_reasoning": "brief explanation"
}}
"""

    text = generate_with_gemini(
        prompt=prompt,
        system_instruction=JUDGE_SYSTEM_PROMPT,
        temperature=0.0,
        max_output_tokens=500,
    )

    result = parse_json_response(text)

    for key in [
        "correctness",
        "completeness",
        "faithfulness",
        "citation_accuracy"
    ]:

        result[key] = float(result[key])

        if not 0 <= result[key] <= 5:
            raise ValueError(
                f"{key} outside 0-5: {result[key]}"
            )

    return result


In [17]:
# ============================================================
# LLM-AS-JUDGE — EVALUATE ALL 26 GENERATED ANSWERS
# ============================================================

import json
import re
import time


# ============================================================
# 1. JUDGE SYSTEM PROMPT
# ============================================================

JUDGE_SYSTEM_PROMPT = """
You are a strict evaluator of a quantitative-finance RAG system.

Evaluate the GENERATED ANSWER against:

1. REFERENCE ANSWER
   - This is the expected answer generated only from the gold source chunks.

2. RETRIEVED CONTEXT
   - This is the evidence actually available to the RAG system.

Score each dimension from 0 to 5.

CORRECTNESS:
5 = fully correct
4 = essentially correct
3 = partially correct
2 = substantial errors or omissions
1 = mostly incorrect
0 = incorrect or irrelevant

COMPLETENESS:
5 = covers all important reference points
4 = covers nearly all important points
3 = covers the main idea
2 = partially covers the answer
1 = minimal coverage
0 = no useful coverage

FAITHFULNESS:
5 = all substantive claims are supported by retrieved context
4 = almost all claims are supported
3 = mixed support
2 = several unsupported claims
1 = mostly unsupported
0 = fabricated or contradictory

CITATION_ACCURACY:
5 = citations correctly support the claims
4 = almost all citations are correct
3 = mixed citation quality
2 = several weak or incorrect citations
1 = mostly unsupported citations
0 = citations unusable

Do not reward writing style.
Do not use outside knowledge.
Evaluate only using the question, reference answer, generated answer,
and retrieved context.

Return JSON only.
"""


# ============================================================
# 2. ROBUST JSON PARSER
# ============================================================

def parse_json_response(text):

    text = text.strip()

    # --------------------------------------------------------
    # Remove markdown code fences
    # --------------------------------------------------------

    text = re.sub(
        r"```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace("```", "").strip()

    # --------------------------------------------------------
    # Locate JSON object
    # --------------------------------------------------------

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(
            "Judge did not return a JSON object."
        )

    json_text = text[start:end + 1]

    # --------------------------------------------------------
    # First attempt: normal JSON
    # --------------------------------------------------------

    try:
        result = json.loads(json_text)

        return result

    except json.JSONDecodeError:
        pass


    # --------------------------------------------------------
    # Fallback:
    # Extract numerical scores directly.
    #
    # This protects against invalid backslashes / LaTeX
    # appearing inside overall_reasoning.
    # --------------------------------------------------------

    patterns = {

        "correctness":
            r'"correctness"\s*:\s*([0-5](?:\.\d+)?)',

        "completeness":
            r'"completeness"\s*:\s*([0-5](?:\.\d+)?)',

        "faithfulness":
            r'"faithfulness"\s*:\s*([0-5](?:\.\d+)?)',

        "citation_accuracy":
            r'"citation_accuracy"\s*:\s*([0-5](?:\.\d+)?)',
    }

    scores = {}

    for key, pattern in patterns.items():

        match = re.search(
            pattern,
            json_text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if not match:

            raise ValueError(
                f"Could not extract judge score: {key}"
            )

        scores[key] = float(
            match.group(1)
        )


    # --------------------------------------------------------
    # Extract reasoning if possible.
    # If reasoning is malformed, simply ignore it.
    # --------------------------------------------------------

    reasoning_match = re.search(
        r'"overall_reasoning"\s*:\s*"(.*?)"',
        json_text,
        flags=re.DOTALL
    )

    if reasoning_match:

        reasoning = reasoning_match.group(1)

    else:

        reasoning = (
            "Scores extracted successfully; "
            "judge reasoning could not be parsed."
        )


    scores["overall_reasoning"] = reasoning

    return scores


# ============================================================
# 3. JUDGE FUNCTION
# ============================================================

def judge_rag_answer(
    query,
    reference_answer,
    generated_answer,
    context,
    citations
):

    prompt = f"""
QUESTION:
{query}

REFERENCE ANSWER:
{reference_answer}

GENERATED ANSWER:
{generated_answer}

GENERATED CITATIONS:
{citations}

RETRIEVED CONTEXT:
{context}

Evaluate the generated answer.

Return exactly this JSON structure:

{{
  "correctness": 0,
  "completeness": 0,
  "faithfulness": 0,
  "citation_accuracy": 0,
  "overall_reasoning": "brief explanation"
}}

Important:
- Scores must be numbers from 0 to 5.
- Do not use outside knowledge.
- Keep overall_reasoning short.
- Return JSON only.
"""

    text = generate_with_gemini(
        prompt=prompt,
        system_instruction=JUDGE_SYSTEM_PROMPT,
        temperature=0.0,
        max_output_tokens=500,
    )

    result = parse_json_response(text)


    # --------------------------------------------------------
    # Validate scores
    # --------------------------------------------------------

    for key in [
        "correctness",
        "completeness",
        "faithfulness",
        "citation_accuracy"
    ]:

        if key not in result:

            raise ValueError(
                f"Missing judge score: {key}"
            )

        result[key] = float(
            result[key]
        )

        if not 0 <= result[key] <= 5:

            raise ValueError(
                f"{key} outside 0-5: {result[key]}"
            )


    return result


# ============================================================
# 4. LOAD REFERENCE ANSWERS
# ============================================================

with open(
    REFERENCE_ANSWERS_PATH,
    "r",
    encoding="utf-8"
) as f:

    reference_answers = json.load(f)


reference_by_id = {
    x["query_id"]: x
    for x in reference_answers
}


if len(reference_by_id) != 26:

    raise ValueError(
        f"Need 26 reference answers; "
        f"found {len(reference_by_id)}."
    )


# ============================================================
# 5. LOAD EXISTING JUDGE RESULTS
# ============================================================

JUDGE_RESULTS_PATH = (
    OUTPUT_DIR / "rag_judge_results.json"
)


if JUDGE_RESULTS_PATH.exists():

    with open(
        JUDGE_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        evaluation_results = json.load(f)

else:

    evaluation_results = []


evaluation_by_id = {
    x["query_id"]: x
    for x in evaluation_results
}


print(
    "Existing judge results:",
    len(evaluation_results)
)

print(
    "Gemini model:",
    LLM_MODEL
)


# ============================================================
# 6. EVALUATE ALL 26
# ============================================================

for i, item in enumerate(
    evaluation_set,
    1
):

    qid = item["query_id"]


    # --------------------------------------------------------
    # Skip already evaluated questions
    # --------------------------------------------------------

    if (
        qid in evaluation_by_id
        and evaluation_by_id[qid].get(
            "status"
        ) == "evaluated"
    ):

        print(
            f"[{i}/26] {qid} "
            f"— already evaluated, skipping."
        )

        continue


    # --------------------------------------------------------
    # Find RAG result
    # --------------------------------------------------------

    rag = next(
        (
            x
            for x in rag_results
            if x["query_id"] == qid
        ),
        None
    )


    reference = reference_by_id[qid]


    print("\n" + "=" * 75)

    print(
        f"[{i}/26] "
        f"{qid} — "
        f"{item['category']}"
    )


    # ========================================================
    # RAG ANSWER UNAVAILABLE
    # ========================================================

    if (
        rag is None
        or rag.get("status") != "success"
    ):

        row = {

            "query_id": qid,

            "category": item["category"],

            "query": item["query"],

            "status": "error",

            "correctness": None,

            "completeness": None,

            "faithfulness": None,

            "citation_accuracy": None,

            "overall_score": None,

            "reasoning":
                "RAG answer failed or is unavailable.",
        }


        evaluation_by_id[qid] = row


        print(
            "Skipped: RAG answer unavailable."
        )


    # ========================================================
    # JUDGE RAG ANSWER
    # ========================================================

    else:

        # ----------------------------------------------------
        # Build retrieved context
        # ----------------------------------------------------

        context_records = (
            build_context_records(
                rag["retrieved_chunks"]
            )
        )

        context = build_context(
            context_records
        )


        try:

            judged = judge_rag_answer(

                query=item["query"],

                reference_answer=
                    reference[
                        "reference_answer"
                    ],

                generated_answer=
                    rag["answer"],

                context=context,

                citations=
                    rag["citations"],
            )


            # ------------------------------------------------
            # Weighted overall score
            # ------------------------------------------------

            overall = (

                0.40 *
                judged["correctness"]

                +

                0.25 *
                judged["completeness"]

                +

                0.25 *
                judged["faithfulness"]

                +

                0.10 *
                judged["citation_accuracy"]
            )


            # ------------------------------------------------
            # Store result
            # ------------------------------------------------

            row = {

                "query_id":
                    qid,

                "category":
                    item["category"],

                "query":
                    item["query"],

                "status":
                    "evaluated",

                "correctness":
                    judged["correctness"],

                "completeness":
                    judged["completeness"],

                "faithfulness":
                    judged["faithfulness"],

                "citation_accuracy":
                    judged["citation_accuracy"],

                "overall_score":
                    overall,

                "reasoning":
                    judged.get(
                        "overall_reasoning",
                        ""
                    ),
            }


            evaluation_by_id[qid] = row


            # ------------------------------------------------
            # Print result
            # ------------------------------------------------

            print(

                f"Correctness="
                f"{row['correctness']:.1f}/5 | "

                f"Completeness="
                f"{row['completeness']:.1f}/5 | "

                f"Faithfulness="
                f"{row['faithfulness']:.1f}/5 | "

                f"Citation="
                f"{row['citation_accuracy']:.1f}/5 | "

                f"Overall="
                f"{row['overall_score']:.2f}/5"
            )


        # ----------------------------------------------------
        # Rate limit / API error
        # ----------------------------------------------------

        except RuntimeError as e:

            print(
                "\nSTOPPED:",
                e
            )

            print(
                "Existing judge results "
                "have been preserved."
            )

            print(
                "Run this cell later "
                "to resume."
            )

            break


        # ----------------------------------------------------
        # Parser / validation error
        # ----------------------------------------------------

        except (
            ValueError,
            json.JSONDecodeError
        ) as e:

            print(
                "\nJUDGE PARSING ERROR:",
                e
            )

            print(
                "This question was not "
                "marked as evaluated."
            )

            print(
                "Fix the parser and rerun "
                "this cell."
            )

            break


    # ========================================================
    # 7. SAVE CHECKPOINT AFTER EVERY QUESTION
    # ========================================================

    evaluation_results = [

        evaluation_by_id[x["query_id"]]

        for x in evaluation_set

        if x["query_id"]
        in evaluation_by_id
    ]


    with open(
        JUDGE_RESULTS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(

            evaluation_results,

            f,

            indent=2,

            ensure_ascii=False
        )


    # --------------------------------------------------------
    # Small delay between successful API calls
    # --------------------------------------------------------

    if i < len(evaluation_set):

        time.sleep(
            SUCCESS_DELAY
        )


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)

print(
    "LLM-AS-JUDGE SUMMARY"
)

print("=" * 75)


print(
    "Judge results saved:",
    len(evaluation_results)
)


evaluated_count = sum(

    1

    for x in evaluation_results

    if x.get("status")
    == "evaluated"
)


print(
    "Successfully evaluated:",
    evaluated_count
)


missing = [

    x["query_id"]

    for x in evaluation_set

    if (
        x["query_id"]
        not in evaluation_by_id

        or

        evaluation_by_id[
            x["query_id"]
        ].get("status")
        != "evaluated"
    )
]


print(
    "Remaining:",
    len(missing)
)


if missing:

    print(
        "Missing:",
        missing
    )

else:

    print(
        "\nAll 26 answers have been evaluated."
    )

Existing judge results: 6
Gemini model: gemini-3.5-flash-lite
[1/26] SEM01 — already evaluated, skipping.
[2/26] SEM02 — already evaluated, skipping.
[3/26] SEM03 — already evaluated, skipping.
[4/26] SEM04 — already evaluated, skipping.
[5/26] SEM05 — already evaluated, skipping.
[6/26] SEM06 — already evaluated, skipping.

[7/26] SEM07 — semantic_factual
Correctness=5.0/5 | Completeness=5.0/5 | Faithfulness=5.0/5 | Citation=5.0/5 | Overall=5.00/5

[8/26] SEM08 — semantic_factual
Correctness=4.0/5 | Completeness=3.0/5 | Faithfulness=5.0/5 | Citation=5.0/5 | Overall=4.10/5

[9/26] SEM09 — semantic_factual
Correctness=5.0/5 | Completeness=5.0/5 | Faithfulness=5.0/5 | Citation=5.0/5 | Overall=5.00/5

[10/26] SEM10 — semantic_factual
Correctness=2.0/5 | Completeness=2.0/5 | Faithfulness=5.0/5 | Citation=5.0/5 | Overall=3.05/5

[11/26] FOR01 — formula_quantitative
Correctness=5.0/5 | Completeness=5.0/5 | Faithfulness=5.0/5 | Citation=3.0/5 | Overall=4.80/5

[12/26] FOR02 — formula_quantita

In [18]:
# ============================================================
# FINAL RAG EVALUATION METRICS
# ============================================================

df = pd.DataFrame(evaluation_results)

valid = df[
    df["status"] == "evaluated"
].copy()

if valid.empty:
    raise ValueError("No successful RAG evaluations.")

metric_cols = [
    "correctness",
    "completeness",
    "faithfulness",
    "citation_accuracy",
    "overall_score"
]

overall_metrics = {
    col: {
        "mean_0_to_5": round(
            float(valid[col].mean()),
            4
        ),
        "mean_percent": round(
            float(valid[col].mean() / 5 * 100),
            2
        )
    }
    for col in metric_cols
}

category_table = (
    valid
    .groupby("category")[metric_cols]
    .mean()
    .round(3)
)

final_metrics = {
    "evaluation_type": "RAG system evaluation only",
    "reference_type": "Gemini-generated from gold source chunks",
    "total_questions": 26,
    "evaluated_questions": int(len(valid)),
    "failed_questions": int(26 - len(valid)),
    "llm_model": LLM_MODEL,
    "score_scale": "0-5",
    "overall_metrics": overall_metrics,
    "category_metrics": category_table.to_dict(
        orient="index"
    ),
    "overall_score_weights": {
        "correctness": 0.40,
        "completeness": 0.25,
        "faithfulness": 0.25,
        "citation_accuracy": 0.10
    }
}

print("=" * 75)
print("FINAL RAG EVALUATION")
print("=" * 75)

for metric, values in overall_metrics.items():

    print(
        f"{metric:22s}: "
        f"{values['mean_0_to_5']:.2f}/5 "
        f"({values['mean_percent']:.2f}%)"
    )

display(category_table)


FINAL RAG EVALUATION
correctness           : 4.50/5 (90.00%)
completeness          : 4.35/5 (86.92%)
faithfulness          : 4.92/5 (98.46%)
citation_accuracy     : 4.58/5 (91.54%)
overall_score         : 4.58/5 (91.50%)


,correctness,completeness,faithfulness,citation_accuracy,overall_score
category,,,,,
cross_document,5.0,5.000,5.000,5.0,5.000
figure_based,5.0,5.000,5.000,5.0,5.000
formula_quantitative,3.5,3.167,4.667,4.0,3.758
multi_chunk_reasoning,5.0,5.000,5.000,5.0,5.000
semantic_factual,4.6,4.400,5.000,4.5,4.640
table_based,5.0,5.000,5.000,5.0,5.000


In [20]:
# ============================================================
# SAVE FINAL ARTIFACTS
# ============================================================

with open(
    RAG_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2,
        ensure_ascii=False
    )

matrix = valid[
    [
        "query_id",
        "category",
        "correctness",
        "completeness",
        "faithfulness",
        "citation_accuracy",
        "overall_score",
    ]
].copy()

matrix["overall_percent"] = (
    matrix["overall_score"] / 5 * 100
)

matrix.to_csv(
    RAG_MATRIX_PATH,
    index=False
)

print("Saved:")
print(" ", REFERENCE_ANSWERS_PATH)
print(" ", RAG_RESULTS_PATH)
print(" ", RAG_METRICS_PATH)
print(" ", RAG_MATRIX_PATH)
print(" ", JUDGE_RESULTS_PATH)


Saved:
  c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\reference_answers.json
  c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_evaluation_results.json
  c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_evaluation_metrics.json
  c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_evaluation_matrix.csv
  c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_judge_results.json


In [21]:
# ============================================================
# DIRECT COMPARISON:
# REFERENCE ANSWER vs RAG GENERATED ANSWER
# ============================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. LOAD EXISTING RESULTS
# ============================================================

with open(
    REFERENCE_ANSWERS_PATH,
    "r",
    encoding="utf-8"
) as f:
    reference_answers = json.load(f)


with open(
    RAG_RESULTS_PATH,
    "r",
    encoding="utf-8"
) as f:
    rag_results = json.load(f)


reference_by_id = {
    x["query_id"]: x
    for x in reference_answers
}

rag_by_id = {
    x["query_id"]: x
    for x in rag_results
}


if len(reference_by_id) != 26:
    raise ValueError(
        f"Expected 26 reference answers, found {len(reference_by_id)}"
    )


if len(rag_by_id) != 26:
    raise ValueError(
        f"Expected 26 RAG answers, found {len(rag_by_id)}"
    )


print("Reference answers:", len(reference_by_id))
print("RAG answers:", len(rag_by_id))


# ============================================================
# 2. LOAD EXISTING EMBEDDING MODEL
# ============================================================

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME
)


# ============================================================
# 3. SIMPLE TOKENIZATION
# ============================================================

def tokenize(text):

    return re.findall(
        r"\b[a-zA-Z0-9]+(?:\.[0-9]+)?\b",
        text.lower()
    )


# ============================================================
# 4. LEXICAL F1
# ============================================================

def token_f1(reference, generated):

    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)

    if not ref_tokens or not gen_tokens:
        return 0.0

    ref_counts = {}

    for token in ref_tokens:
        ref_counts[token] = (
            ref_counts.get(token, 0) + 1
        )

    gen_counts = {}

    for token in gen_tokens:
        gen_counts[token] = (
            gen_counts.get(token, 0) + 1
        )

    overlap = 0

    for token in gen_counts:

        if token in ref_counts:

            overlap += min(
                gen_counts[token],
                ref_counts[token]
            )

    precision = (
        overlap / len(gen_tokens)
    )

    recall = (
        overlap / len(ref_tokens)
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# 5. PREPARE 26 ANSWERS
# ============================================================

query_ids = [
    item["query_id"]
    for item in evaluation_set
]


reference_texts = []
generated_texts = []

valid_ids = []


for qid in query_ids:

    reference = reference_by_id[qid]
    rag = rag_by_id[qid]

    if rag.get("status") != "success":
        continue

    reference_texts.append(
        reference["reference_answer"]
    )

    generated_texts.append(
        rag["answer"]
    )

    valid_ids.append(qid)


print(
    "Answers available for direct comparison:",
    len(valid_ids)
)


if len(valid_ids) != 26:
    raise ValueError(
        "Direct comparison requires all 26 successful RAG answers."
    )


# ============================================================
# 6. SEMANTIC SIMILARITY
# ============================================================

reference_embeddings = embedding_model.encode(
    reference_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)


generated_embeddings = embedding_model.encode(
    generated_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)


semantic_scores = np.sum(
    reference_embeddings *
    generated_embeddings,
    axis=1
)


# ============================================================
# 7. BUILD DIRECT COMPARISON RESULTS
# ============================================================

comparison_results = []


for idx, qid in enumerate(valid_ids):

    reference = reference_by_id[qid]
    rag = rag_by_id[qid]

    reference_answer = (
        reference["reference_answer"]
    )

    generated_answer = (
        rag["answer"]
    )

    semantic_similarity = float(
        semantic_scores[idx]
    )

    lexical_f1 = token_f1(
        reference_answer,
        generated_answer
    )

    reference_tokens = len(
        tokenize(reference_answer)
    )

    generated_tokens = len(
        tokenize(generated_answer)
    )

    if reference_tokens > 0:

        length_ratio = (
            generated_tokens /
            reference_tokens
        )

    else:

        length_ratio = 0.0


    comparison_results.append({

        "query_id": qid,

        "category":
            reference["category"],

        "query":
            reference["query"],

        "semantic_similarity":
            semantic_similarity,

        "lexical_f1":
            lexical_f1,

        "reference_token_count":
            reference_tokens,

        "generated_token_count":
            generated_tokens,

        "length_ratio":
            length_ratio,

        "reference_answer":
            reference_answer,

        "generated_answer":
            generated_answer
    })


# ============================================================
# 8. DATAFRAME
# ============================================================

comparison_df = pd.DataFrame(
    comparison_results
)


# ============================================================
# 9. SAVE RESULTS
# ============================================================

DIRECT_COMPARISON_JSON = (
    OUTPUT_DIR /
    "rag_direct_comparison_results.json"
)

DIRECT_COMPARISON_CSV = (
    OUTPUT_DIR /
    "rag_direct_comparison_results.csv"
)


with open(
    DIRECT_COMPARISON_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        comparison_results,
        f,
        indent=2,
        ensure_ascii=False
    )


comparison_df.to_csv(
    DIRECT_COMPARISON_CSV,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 10. FINAL DIRECT COMPARISON METRICS
# ============================================================

mean_semantic = (
    comparison_df[
        "semantic_similarity"
    ].mean()
)

mean_lexical_f1 = (
    comparison_df[
        "lexical_f1"
    ].mean()
)

mean_length_ratio = (
    comparison_df[
        "length_ratio"
    ].mean()
)


print("\n" + "=" * 70)
print("DIRECT REFERENCE vs RAG COMPARISON")
print("=" * 70)

print(
    f"Semantic Similarity : "
    f"{mean_semantic:.4f} "
    f"({mean_semantic * 100:.2f}%)"
)

print(
    f"Lexical F1          : "
    f"{mean_lexical_f1:.4f} "
    f"({mean_lexical_f1 * 100:.2f}%)"
)

print(
    f"Average Length Ratio: "
    f"{mean_length_ratio:.4f}"
)

print(
    "\nQuestions compared:",
    len(comparison_df)
)

print(
    "\nSaved:"
)

print(
    DIRECT_COMPARISON_JSON
)

print(
    DIRECT_COMPARISON_CSV
)


# ============================================================
# 11. CATEGORY-WISE DIRECT COMPARISON
# ============================================================

category_comparison = (
    comparison_df
    .groupby("category")
    [
        [
            "semantic_similarity",
            "lexical_f1"
        ]
    ]
    .mean()
    .round(4)
)


print("\n" + "=" * 70)
print("CATEGORY-WISE DIRECT COMPARISON")
print("=" * 70)

display(category_comparison)

Reference answers: 26
RAG answers: 26


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2126.13it/s]


Embedding model: BAAI/bge-small-en-v1.5
Answers available for direct comparison: 26


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.88s/it]


DIRECT REFERENCE vs RAG COMPARISON
Semantic Similarity : 0.9192 (91.92%)
Lexical F1          : 0.5189 (51.89%)
Average Length Ratio: 1.5956

Questions compared: 26

Saved:
c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_direct_comparison_results.json
c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\rag_direct_comparison_results.csv

CATEGORY-WISE DIRECT COMPARISON


,semantic_similarity,lexical_f1
category,,
cross_document,0.9832,0.7756
figure_based,0.8668,0.4703
formula_quantitative,0.8877,0.3932
multi_chunk_reasoning,0.9780,0.6921
semantic_factual,0.9245,0.5157
table_based,0.9388,0.5947


## Final reported RAG metrics

Primary metrics:
1. Answer Correctness
2. Answer Completeness
3. Faithfulness / Groundedness
4. Citation Accuracy

Overall RAG Score:

40% Correctness + 25% Completeness + 25% Faithfulness + 10% Citation Accuracy

Retrieval metrics are intentionally excluded.

Reference answers are generated from the frozen gold chunks using Gemini and are
not human-verified unless manually reviewed.
